# Mp3 Transcribtion

-----------

In [1]:
choose_your_mp3_path="./files"

In [2]:
import os
import shutil
from pathlib import Path
from typing import List, Dict, Optional, Tuple
import json
from langchain_community.llms import Ollama
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

import ipdb

BASE_DIR = Path(".")  # Your mp3 root directory
SERIES_PREFIX = "SERIES_"  # Prefix for multi-part mp3 folders
MODEL_NAME = "mistral-small3.2:24b"  # or "mistral:latest"

class CategoryMatch(BaseModel):
    category_path: str = Field(
        description="The relative path to the category folder"
    )
    confidence: float = Field(
        description="Confidence score between 0 and 1"
    )
    reasoning: str = Field(
        description="Brief explanation of why this category fits"
    )

def is_series_folder(folder_name: str) -> bool:
    return folder_name.startswith(SERIES_PREFIX)

def get_category_structure(base_dir: Path, max_depth: int = 3) -> Dict:
    
    def explore_dir(current_dir: Path, current_depth: int = 0, rel_path: str = ""):
        if current_depth > max_depth:
            return {}
        
        result = {
            "path": rel_path,
            "sample_files": [],
            "subcategories": {}
        }
        
        try:
            items = list(current_dir.iterdir())
        except PermissionError:
            return result
        
        mp3_files = [
            f.name for f in items 
            if f.is_file() and f.suffix.lower() == '.mp3'
        ]
        result["sample_files"] = mp3_files[:5]  # Limit to 5 samples
        
        subdirs = [
            d for d in items 
            if d.is_dir() and not is_series_folder(d.name)
        ]
        
        for subdir in subdirs:
            new_rel_path = f"{rel_path}/{subdir.name}" if rel_path else subdir.name
            result["subcategories"][subdir.name] = explore_dir(
                subdir, 
                current_depth + 1, 
                new_rel_path
            )
        
        return result
    
    return explore_dir(base_dir)


def format_category_structure(structure: Dict, indent: int = 0) -> str:
    
    def format_tree_node(node_data: Dict, depth: int = 0, is_last: bool = True, prefix: str = "") -> List[str]:
        lines = []
        
        path = node_data.get("path", "")
        if path:
            folder_name = path.split("/")[-1]
        else:
            folder_name = "ROOT"
        
        # Don't render the ROOT node itself, just its children
        if depth == 0:
            subcategories = node_data.get("subcategories", {})
            subcat_items = list(subcategories.items())
            for i, (subcat_name, subcat_data) in enumerate(subcat_items):
                is_last_subcat = (i == len(subcat_items) - 1)
                lines.extend(format_tree_node(
                    subcat_data,
                    depth + 1,
                    is_last_subcat,
                    ""
                ))
            return lines
        
        connector = "└── " if is_last else "├── "
        current_prefix = prefix + ("    " if is_last else "│   ")
        
        sample_files = node_data.get("sample_files", [])
        if sample_files:
            sample_info = f" ({len(sample_files)} files)"
            if len(sample_files) > 0:
                clean_samples = [f.replace('.mp3', '').replace('_', ' ')[:40] for f in sample_files[:2]]
                sample_info += f" - e.g., {', '.join(clean_samples)}"
        else:
            sample_info = ""
        
        folder_line = f"{prefix}{connector}{folder_name}{sample_info}"
        lines.append(folder_line)
        
        subcategories = node_data.get("subcategories", {})
        subcat_items = list(subcategories.items())
        
        for i, (subcat_name, subcat_data) in enumerate(subcat_items):
            is_last_subcat = (i == len(subcat_items) - 1)
            lines.extend(format_tree_node(
                subcat_data,
                depth + 1,
                is_last_subcat,
                current_prefix
            ))
        
        return lines
    
    tree_lines = format_tree_node(structure)
    return "\n".join(tree_lines)
    
def get_uncategorized_files(base_dir: Path) -> List[Path]:
    return [
        f for f in base_dir.iterdir()
        if f.is_file() and f.suffix.lower() == '.mp3'
    ]

def find_companion_txt(mp3_file: Path) -> Optional[Path]:
    txt_file = mp3_file.with_suffix('.txt')
    return txt_file if txt_file.exists() else None

def create_classification_prompt(mp3_title: str, category_structure: Dict) -> str:
    
    formatted_structure = format_category_structure(category_structure)
    
    prompt = f"""# Mp3 Classification Analysis

You are an expert at categorizing Mp3 files based on their titles and existing category structure.

## Category Structure:
{formatted_structure}

## mp3 to Categorize:
**Title:** {mp3_title}

## Analysis Required:

Please analyze the mp3 title and determine which category folder it best fits into.

Consider the following factors:
1. **Geographic relevance** (e.g., Africa, Asia, Europe)
2. **Thematic content** (e.g., wars, historical periods, specific countries)
3. **Similar mp3 files** already in each category
4. **Specificity** - Go for the most specific category possible (prefer Africa/Libya over just Africa)

## Task:
Provide your classification with:
- The exact category path where this mp3 should be placed
- A confidence score between 0 and 1
- A brief reasoning for your decision

Please respond in the following JSON format:
{{
    "category_path": "exact/path/to/category",
    "confidence": 0.85,
    "reasoning": "Brief explanation of why this category fits"
}}
"""
    
    return prompt

def create_classification_parser():
    return PydanticOutputParser(pydantic_object=CategoryMatch)

def call_llm_with_parser(prompt: str, parser: PydanticOutputParser, model_name: str = MODEL_NAME):
    llm = Ollama(model=model_name, temperature=0.1)
    
    full_prompt = prompt + "\n\n" + parser.get_format_instructions()
    
    try:
        response = llm.invoke(full_prompt)
        result = parser.parse(response)
        return result
    except Exception as e:
        print(f"Error parsing LLM response: {e}")
        print(f"Raw response: {response}")
        # Return fallback result
        return CategoryMatch(
            category_path="",
            confidence=0.0,
            reasoning="Classification failed"
        )

class mp3Classifier:
    def __init__(self, model_name: str = MODEL_NAME):
        self.model_name = model_name
    
    def classify(self, mp3_title: str, category_structure: Dict) -> CategoryMatch:
        """Classify a mp3 into a category"""
        
        prompt = create_classification_prompt(mp3_title, category_structure)
        
        parser = create_classification_parser()
        
        # Debug point
        #ipdb.set_trace()
        
        result = call_llm_with_parser(prompt, parser, self.model_name)
        
        return result

def move_mp3(
    mp3_file: Path,
    target_category: str,
    base_dir: Path,
    dry_run: bool = False
) -> Tuple[bool, str]:
    target_dir = base_dir / target_category
    
    if not target_dir.exists():
        return False, f"Target directory does not exist: {target_dir}"
    
    txt_file = find_companion_txt(mp3_file)
    files_to_move = [mp3_file]
    if txt_file:
        files_to_move.append(txt_file)
    
    if dry_run:
        message = f"[DRY RUN] Would move:\n"
        for file in files_to_move:
            message += f"  {file.name} -> {target_category}/\n"
        return True, message
    
    try:
        for file in files_to_move:
            target_path = target_dir / file.name
            
            if target_path.exists():
                return False, f"File already exists: {target_path}"
            
            shutil.move(str(file), str(target_path))
        
        return True, f"Moved {mp3_file.name} (+ txt) to {target_category}/"
    
    except Exception as e:
        return False, f"Error moving files: {e}"

def categorize_documentaries(
    base_dir: Path = BASE_DIR,
    dry_run: bool = True,
    interactive: bool = True,
    auto_mode: bool = False,
    confidence_threshold: float = 0.7
):
    print(f"\nBase directory: {base_dir.absolute()}")
    print(f"Mode: {'DRY RUN' if dry_run else 'LIVE'}")
    print(f"Auto-mode: {'ENABLED - No prompts' if auto_mode else 'DISABLED'}")
    print(f"Interactive: {interactive and not auto_mode}")
    print(f"Confidence threshold: {confidence_threshold}\n")
    
    print("Analyzing category structure...")
    category_structure = get_category_structure(base_dir)
    
    print("Initializing LLM classifier...")
    classifier = mp3Classifier()
    
    print("Finding uncategorized documentaries...")
    uncategorized = get_uncategorized_files(base_dir)
    print(f"Found {len(uncategorized)} uncategorized documentaries\n")
    
    if not uncategorized:
        print("All documentaries are already categorized!")
        return
    
    results = {
        "moved": [],
        "skipped": [],
        "failed": []
    }
    
    yes_to_all = auto_mode 
    
    for idx, mp3_file in enumerate(uncategorized, 1):
        print(f"\n[{idx}/{len(uncategorized)}] Processing: {mp3_file.name}")
        print("-" * 80)
        
        classification = classifier.classify(mp3_file.stem, category_structure)
        
        print(f"Suggested category: {classification.category_path}")
        print(f"Confidence: {classification.confidence:.2f}")
        print(f"Reasoning: {classification.reasoning}")
        
        should_move = False
        
        if classification.confidence >= confidence_threshold:
            if yes_to_all:
                should_move = True
                print("✓ Auto-accepting (yes-to-all mode)")
            elif interactive:
                response = input(f"\nMove to '{classification.category_path}'? [Y/n/a(yes to all)/s(skip all)]: ").strip().lower()
                if response == 's':
                    print("Skipping all remaining files")
                    break
                elif response == 'a':
                    print("✓ Enabling yes-to-all mode for remaining files")
                    yes_to_all = True
                    should_move = True
                else:
                    should_move = response in ['', 'y', 'yes']
            else:
                should_move = True
        else:
            print(f"Confidence below threshold ({confidence_threshold})")
            if not yes_to_all and interactive:
                response = input(f"Move anyway? [y/N/a(yes to all)]: ").strip().lower()
                if response == 'a':
                    print("✓ Enabling yes-to-all mode for remaining files")
                    yes_to_all = True
                    should_move = True
                else:
                    should_move = response in ['y', 'yes']
            elif yes_to_all:
                print("Skipped (below confidence threshold)")
        
        if should_move:
            success, message = move_mp3(
                mp3_file,
                classification.category_path,
                base_dir,
                dry_run=dry_run
            )
            
            if success:
                print(f"✓  {message}")
                results["moved"].append((mp3_file.name, classification.category_path))
            else:
                print(f"X {message}")
                results["failed"].append((mp3_file.name, message))
        else:
            print("Skipped")
            results["skipped"].append(mp3_file.name)
    
    print("\n" + "=" * 80)
    print("SUMMARY")
    print("=" * 80)
    print(f"    Moved: {len(results['moved'])}")
    print(f"    Skipped: {len(results['skipped'])}")
    print(f"    Failed: {len(results['failed'])}")
    
    if results["moved"]:
        print("\nMoved files:")
        for filename, category in results["moved"]:
            print(f"  • {filename} → {category}")
    
    if results["failed"]:
        print("\nFailed files:")
        for filename, error in results["failed"]:
            print(f"  • {filename}: {error}")

def mark_mp3_series(base_dir: Path):
    print(f"This tool will help you mark multi-part mp3 folders.")
    print(f"They will be prefixed with '{SERIES_PREFIX}'\n")
    
    candidates = []
    
    for category_dir in base_dir.iterdir():
        if not category_dir.is_dir() or is_series_folder(category_dir.name):
            continue
        
        for subdir in category_dir.rglob("*"):
            if not subdir.is_dir() or is_series_folder(subdir.name):
                continue
            
            files = list(subdir.glob("*.mp3"))
            if len(files) >= 2:
                has_episode_pattern = any(
                    '(' in f.stem and '_' in f.stem and ')' in f.stem
                    for f in files
                )
                if has_episode_pattern:
                    candidates.append(subdir)
    
    if not candidates:
        print("No multi-part mp3 folders found!")
        return
    
    print(f"Found {len(candidates)} potential mp3 series:\n")
    
    for idx, folder in enumerate(candidates, 1):
        print(f"\n[{idx}/{len(candidates)}] {folder.relative_to(base_dir)}")
        files = sorted(folder.glob("*.mp3"))
        print(f"  Contains {len(files)} episodes:")
        for f in files[:3]:
            print(f"    • {f.name}")
        if len(files) > 3:
            print(f"    ... and {len(files) - 3} more")
        
        response = input("  Mark as mp3 series? [Y/n]: ").strip().lower()
        
        if response in ['', 'y', 'yes']:
            new_name = SERIES_PREFIX + folder.name
            new_path = folder.parent / new_name
            folder.rename(new_path)
            print(f"    Renamed to: {new_name}")
        else:
            print("     Skipped")

In [3]:
from pathlib import Path

root_path = Path(choose_your_mp3_path)

categorize_documentaries(
        base_dir=root_path,
        confidence_threshold=0.8,
        dry_run=False,
        auto_mode=True
    )


Base directory: /Users/antonio/Projekte/sofakles/Transkribtion/files
Mode: LIVE
Auto-mode: ENABLED - No prompts
Interactive: False
Confidence threshold: 0.8

Analyzing category structure...
Initializing LLM classifier...
Finding uncategorized documentaries...
Found 3 uncategorized documentaries


[1/3] Processing: Adventures of Huckleberry Finn (1_2).mp3
--------------------------------------------------------------------------------


/var/folders/g6/jf2_7bv15b1gjblpbf6_561c0000gn/T/ipykernel_56454/4235643399.py:182: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(model=model_name, temperature=0.1)


Suggested category: Books/American classics
Confidence: 0.95
Reasoning: 'Adventures of Huckleberry Finn' is a well-known classic novel by Mark Twain, which is an American author. The title clearly fits within the 'American classics' subcategory under 'Books'.
✓ Auto-accepting (yes-to-all mode)
✓  Moved Adventures of Huckleberry Finn (1_2).mp3 (+ txt) to Books/American classics/

[2/3] Processing: A Brief History of Liberia and Africa's Iron Lady _ Ellen Johnson Sirleaf.mp3
--------------------------------------------------------------------------------
Suggested category: African politics
Confidence: 0.95
Reasoning: The title 'A Brief History of Liberia and Africa's Iron Lady _ Ellen Johnson Sirleaf' clearly pertains to African politics. It discusses Liberia, an African country, and Ellen Johnson Sirleaf, who is known for her political role as the President of Liberia. The content is directly related to political history and leadership in Africa, making 'African politics' the most appr

In [4]:
from pathlib import Path

root_path = Path(choose_your_mp3_path)

mark_mp3_series(root_path)

This tool will help you mark multi-part mp3 folders.
They will be prefixed with 'SERIES_'

Found 1 potential mp3 series:


[1/1] Books/American classics
  Contains 2 episodes:
    • Adventures of Huckleberry Finn (1_2).mp3
    • Adventures of Huckleberry Finn (2_2).mp3


  Mark as mp3 series? [Y/n]:  y


    Renamed to: SERIES_American classics


-----------------

In [5]:
import os
import whisper
from pathlib import Path

In [6]:
def transcribe_with_whisper(audio_file):
    print("Load whisper model")
    model = whisper.load_model("small")  # options: tiny, base, small, medium, large
    
    print("Transcribe with whisper...")
    result = model.transcribe(audio_file)
    
    return {
        'text': result['text'],
        'language': result['language'],
        'segments': result['segments']
    }

In [7]:
def find_mp3_files(root_path):
    mp3_files = []
    for root, dirs, files in os.walk(root_path):
        for file in files:
            if file.lower().endswith('.mp3'):
                mp3_files.append(os.path.join(root, file))
    return mp3_files

In [8]:
def get_txt_path(mp3_path):
    return os.path.splitext(mp3_path)[0] + '.txt'

def txt_file_exists(mp3_path):
    txt_path = get_txt_path(mp3_path)
    return os.path.exists(txt_path)

In [9]:
def save_transcription(mp3_path, transcription_result):
    txt_path = get_txt_path(mp3_path)
    
    try:
        with open(txt_path, 'w', encoding='utf-8') as f:
            f.write(f"file: {os.path.basename(mp3_path)}\n")
            f.write(f"language: {transcription_result['language']}\n")
            f.write(f"{'='*50}\n\n") #example
            f.write(transcription_result['text'])
        
        print(f"Saved transcription in file: {txt_path}")
        return True
    except Exception as e:
        print(f"Error while saving the transcription: {e}")
        return False

In [10]:
import re
from pathlib import Path

def extract_arte_title_from_path(file_path):
    
    filename = Path(file_path).stem
    
    patterns_to_remove = [
        r'[_\s]*Doku[_\s]*HD.*$'  # Remove "_Doku HD" or " Doku HD" and everything after
    ]
    
    for pattern in patterns_to_remove:
        filename = re.sub(pattern, '', filename, flags=re.IGNORECASE)
    
    name = filename.replace('_', ' ')
    
    name = re.sub(r'\s+', ' ', name).strip()
    
    return name

test_path = "/this/is/an/example/Michail Gorbatschow Doku HD.txt"
title = extract_arte_title_from_path(test_path)
print(f"Full path: {test_path}")
print(f"Extracted title: '{title}'")

Full path: /this/is/an/example/Michail Gorbatschow Doku HD.txt
Extracted title: 'Michail Gorbatschow'


In [11]:
import time

def transcribe_documentaries(root_path):
    
    if not os.path.exists(root_path):
        print(f"Path does not exists: {root_path}")
        return
    
    print(f"Looking for files in: {root_path}")
    
    mp3_files = find_mp3_files(root_path)
    
    if not mp3_files:
        print("No mp3s found.")
        return
    
    print(f"{len(mp3_files)} mp3s found.")
    
    print("Load whisper model")
    model = whisper.load_model("small")  # options: tiny, base, small, medium, large
    
    # Statistiken
    processed = 0
    skipped = 0
    errors = 0
    
    for i, mp3_path in enumerate(mp3_files, 1):
        print(f"\n [{i}/{len(mp3_files)}] process: {mp3_path}")
        
        #if (processed % 5 == 0) and (processed > 0):
            #print("Taking 10-minute break...")
            #time.sleep(600)  # break from processing
        
        if txt_file_exists(mp3_path):
            print(f"skip (file existists already): {get_txt_path(mp3_path)}")
            skipped += 1
            continue
        
        try:
            mp3_title = extract_arte_title_from_path(mp3_path)
            prompt = f"mp3 file with the name: '{mp3_title}'. Transcribe only the english voices, ignore backround voices in other languages."
            print("Transcribe with whisper")
            result = model.transcribe(mp3_path,
                                      language="en",
                                      initial_prompt=prompt
                                      #temperature=0.0,  # More deterministic, less confusion
                                      #no_speech_threshold=0.7,  # Helps ignore background voices
                                      #compression_ratio_threshold=2.0, #Stricter quality control
                                      #logprob_threshold=-0.8 #Higher confidence requirement
                                     )
            
            whisper_result = {
                'text': result['text'],
                'language': result['language'],
                'segments': result['segments']
            }
            
            print(f"Transcribe result:")
            print(f"   language: {whisper_result['language']}")
            print(f"   text (preview): {whisper_result['text'][:100]}...")
            
            if save_transcription(mp3_path, whisper_result):
                processed += 1
            else:
                errors += 1
                
        except Exception as e:
            print(f"Error while transcribing the file {mp3_path}: {e}")
            errors += 1
    
    print(f"\n Done!")
    print(f"    processed: {processed}")
    print(f"    skipped: {skipped}")
    print(f"    errors: {errors}")
    print(f"    total mp3s: {len(mp3_files)}")


In [12]:
mp3_path = choose_your_mp3_path
import warnings
warnings.filterwarnings("ignore", message="FP16 is not supported on CPU; using FP32 instead")
transcribe_documentaries(mp3_path)

Looking for files in: ./files
3 mp3s found.
Load whisper model

 [1/3] process: ./files/African politics/A Brief History of Liberia and Africa's Iron Lady _ Ellen Johnson Sirleaf.mp3
Transcribe with whisper
Transcribe result:
   language: en
   text (preview):  With its rocky history of violent civil wars, military crews and strongman warlords, very few outsi...
Saved transcription in file: ./files/African politics/A Brief History of Liberia and Africa's Iron Lady _ Ellen Johnson Sirleaf.txt

 [2/3] process: ./files/Books/SERIES_American classics/Adventures of Huckleberry Finn (1_2).mp3
Transcribe with whisper
Transcribe result:
   language: en
   text (preview):  This is LibriVox Recording. All LibriVox recordings are in the public domain. For more information ...
Saved transcription in file: ./files/Books/SERIES_American classics/Adventures of Huckleberry Finn (1_2).txt

 [3/3] process: ./files/Books/SERIES_American classics/Adventures of Huckleberry Finn (2_2).mp3
Transcribe with wh

------------

In [ ]:
import subprocess
import time
import psycopg2
from psycopg2 import sql
from psycopg2.extensions import ISOLATION_LEVEL_AUTOCOMMIT
import pandas as pd
import logging

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Database configuration
DB_CONFIG = {
    'host': 'localhost',
    'database': 'mp3files_db',
    'user': 'postgres',
    'password': 'supersafepswd',
    'port': '5432'
}

# Configuration for connecting to default postgres database
DB_CONFIG_DEFAULT = {
    'host': 'localhost',
    'database': 'postgres',  # Connect to default postgres database
    'user': 'postgres',
    'password': 'pswd123',
    'port': '5432'
}

FOLDER_PATH = choose_your_mp3_path

def setup_postgres_docker():
    
    try:
        result = subprocess.run(['docker', '--version'], 
                              capture_output=True, text=True, check=True)
        logger.info(f"Docker found: {result.stdout.strip()}")
    except (subprocess.CalledProcessError, FileNotFoundError):
        logger.error("Docker not found. Please install Docker Desktop first.")
        return False
    
    try:
        subprocess.run(['docker', 'stop', 'postgres-notebook'], 
                      capture_output=True, check=False)
        subprocess.run(['docker', 'rm', 'postgres-notebook'], 
                      capture_output=True, check=False)
        logger.info("Cleaned up existing containers")
    except:
        pass
    
    # Start PostgreSQL container (removed POSTGRES_DB to avoid conflicts)
    docker_cmd = [
        'docker', 'run', '--name', 'postgres-notebook',
        '-e', f'POSTGRES_PASSWORD={DB_CONFIG["password"]}',
        '-p', f'{DB_CONFIG["port"]}:5432',
        '-d', 'postgres:15'
    ]
    
    try:
        result = subprocess.run(docker_cmd, capture_output=True, text=True, check=True)
        logger.info("PostgreSQL container started successfully")
        
        # Wait for PostgreSQL to be ready
        logger.info("Waiting for PostgreSQL to be ready...")
        time.sleep(10)
        
        return True
    except subprocess.CalledProcessError as e:
        logger.error(f"Failed to start PostgreSQL container: {e.stderr}")
        return False

def create_database_if_not_exists():
    """Create the database if it doesn't exist"""
    try:
        conn = psycopg2.connect(**DB_CONFIG_DEFAULT)
        conn.set_isolation_level(ISOLATION_LEVEL_AUTOCOMMIT)
        cursor = conn.cursor()
        
        cursor.execute(
            "SELECT 1 FROM pg_catalog.pg_database WHERE datname = %s",
            (DB_CONFIG['database'],)
        )
        exists = cursor.fetchone()
        
        if not exists:
            cursor.execute(
                sql.SQL("CREATE DATABASE {}").format(
                    sql.Identifier(DB_CONFIG['database'])
                )
            )
            logger.info(f"Database '{DB_CONFIG['database']}' created successfully")
        else:
            logger.info(f"Database '{DB_CONFIG['database']}' already exists")
        
        cursor.close()
        conn.close()
        return True
        
    except Exception as e:
        logger.error(f"Failed to create database: {e}")
        return False

def test_connection():
    """Test the database connection"""
    try:
        conn = psycopg2.connect(**DB_CONFIG)
        cursor = conn.cursor()
        cursor.execute("SELECT version();")
        version = cursor.fetchone()
        logger.info(f"Connected to PostgreSQL: {version[0]}")
        cursor.close()
        conn.close()
        return True
    except Exception as e:
        logger.error(f"Connection failed: {e}")
        return False

def wait_for_postgres():
    """Wait for PostgreSQL to be ready by trying to connect to default database"""
    max_attempts = 30
    for attempt in range(max_attempts):
        try:
            conn = psycopg2.connect(**DB_CONFIG_DEFAULT)
            conn.close()
            logger.info("PostgreSQL is ready!")
            return True
        except Exception as e:
            if attempt < max_attempts - 1:
                logger.info(f"Waiting for PostgreSQL... (attempt {attempt + 1}/{max_attempts})")
                time.sleep(2)
            else:
                logger.error(f"PostgreSQL not ready after {max_attempts} attempts: {e}")
                return False
    return False

# Run the setup
if setup_postgres_docker():
    if wait_for_postgres():
        if create_database_if_not_exists():
            if test_connection():
                print("PostgreSQL is ready to use!")
            else:
                print("PostgreSQL and database created but connection failed")
        else:
            print("Failed to create database")
    else:
        print("PostgreSQL container started but server not ready")
else:
    print("Failed to start PostgreSQL")

In [14]:
def create_table(cursor):
    """Create table to store text file data"""
    create_table_query = """
    CREATE TABLE IF NOT EXISTS text_files (
        id SERIAL PRIMARY KEY,
        filename VARCHAR(255) NOT NULL,
        file_path VARCHAR(500) NOT NULL,
        relative_path VARCHAR(500) NOT NULL,
        content TEXT,
        file_size INTEGER,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    );
    """
    cursor.execute(create_table_query)
    logger.info("Table 'text_files' created or already exists")

def read_text_file(file_path):
    """Read text file content with error handling"""
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            content = file.read()
        return content
    except UnicodeDecodeError:
        try:
            with open(file_path, 'r', encoding='latin-1') as file:
                content = file.read()
            return content
        except Exception as e:
            logger.error(f"Error reading file {file_path}: {e}")
            return None
    except Exception as e:
        logger.error(f"Error reading file {file_path}: {e}")
        return None

def get_all_text_files(folder_path):
    """Recursively get all text files from folder and subfolders"""
    text_files = []
    folder_path = Path(folder_path)
    
    text_extensions = {'.txt', '.text', '.log', '.md', '.csv', '.json', '.xml'}
    
    for file_path in folder_path.rglob('*'):
        if file_path.is_file() and file_path.suffix.lower() in text_extensions:
            text_files.append(file_path)
    
    logger.info(f"Found {len(text_files)} text files")
    return text_files

def load_files_to_db(folder_path, db_config):
    """Main function to load all text files into PostgreSQL database"""
    
    try:
        conn = psycopg2.connect(**db_config)
        cursor = conn.cursor()
        logger.info("Connected to PostgreSQL database")
    except Exception as e:
        logger.error(f"Error connecting to database: {e}")
        return
    
    try:
        create_table(cursor)
        conn.commit()
        
        text_files = get_all_text_files(folder_path)
        
        successful_inserts = 0
        failed_inserts = 0
        
        for file_path in text_files:
            try:
                content = read_text_file(file_path)
                if content is None:
                    failed_inserts += 1
                    continue
                
                filename = file_path.name
                full_path = str(file_path.absolute())
                relative_path = str(file_path.relative_to(folder_path))
                file_size = file_path.stat().st_size
                
                insert_query = """
                INSERT INTO text_files (filename, file_path, relative_path, content, file_size)
                VALUES (%s, %s, %s, %s, %s)
                """
                
                cursor.execute(insert_query, (filename, full_path, relative_path, content, file_size))
                successful_inserts += 1
                
                if successful_inserts % 10 == 0:  # Log progress every 10 files
                    logger.info(f"Processed {successful_inserts} files...")
                    
            except Exception as e:
                logger.error(f"Error processing file {file_path}: {e}")
                failed_inserts += 1
                continue
        
        conn.commit()
        
        logger.info(f"Loading complete!")
        logger.info(f"Successfully inserted: {successful_inserts} files")
        logger.info(f"Failed to insert: {failed_inserts} files")
        
    except Exception as e:
        logger.error(f"Error during processing: {e}")
        conn.rollback()
    
    finally:
        cursor.close()
        conn.close()
        logger.info("Database connection closed")

In [15]:
load_files_to_db(FOLDER_PATH, DB_CONFIG)

INFO:__main__:Connected to PostgreSQL database
INFO:__main__:Table 'text_files' created or already exists
INFO:__main__:Found 3 text files
INFO:__main__:Loading complete!
INFO:__main__:Successfully inserted: 3 files
INFO:__main__:Failed to insert: 0 files
INFO:__main__:Database connection closed


In [ ]:
from llama_index.core import Document
from llama_index.core.node_parser import SemanticSplitterNodeParser, SimpleFileNodeParser
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings
import psycopg2
from typing import List
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

embed_model = HuggingFaceEmbedding(
    model_name="Snowflake/snowflake-arctic-embed-m",
    trust_remote_code=True
)

Settings.embed_model = embed_model

semantic_splitter = SemanticSplitterNodeParser(
    buffer_size=1,
    breakpoint_percentile_threshold=95,
    embed_model=embed_model,
)

simple_splitter = SimpleFileNodeParser()

def load_transcripts_from_postgres() -> List[Document]:
    try:
        conn = psycopg2.connect(**DB_CONFIG)
        cursor = conn.cursor()
        
        cursor.execute("""
            SELECT id, filename, file_path, relative_path, content, file_size, created_at
            FROM text_files
            WHERE content IS NOT NULL
            ORDER BY filename, id
            LIMIT 10
        """)
        
        documents = []
        for row in cursor.fetchall():
            doc_id, filename, file_path, relative_path, content, file_size, created_at = row
            
            doc = Document(
                text=content,
                metadata={
                    "documentary_name": filename,
                    "source_id": doc_id,
                    "file_path": file_path,
                    "relative_path": relative_path,
                    "file_size": file_size,
                    "created_at": str(created_at),
                },
                id_=str(doc_id)
            )
            documents.append(doc)
        
        cursor.close()
        conn.close()
        
        logger.info(f"Loaded {len(documents)} documents from db")
        return documents
        
    except Exception as e:
        logger.error(f"Failed to load transcripts: {e}")
        raise

def chunk_transcripts(documents: List[Document]) -> List:
    """
    Chunk documents using semantic splitting.
    LlamaIndex automatically handles node relationships (prev/next).
    """
    logger.info(f"Chunking {len(documents)} documents...")
    nodes = semantic_splitter.get_nodes_from_documents(documents)
    logger.info(f"Created {len(nodes)} semantic chunks")
    return nodes


In [17]:
documents = load_transcripts_from_postgres()

# Chunk them
nodes = chunk_transcripts(documents)

# Inspect results
print(f"\nCreated {len(nodes)} chunks from {len(documents)} documents")

if nodes:
    sample_node = nodes[5] if len(nodes) > 5 else nodes[0]
    print(f"\nSample chunk:")
    print(f"Text preview: {sample_node.text[:200]}...")
    print(f"Metadata: {sample_node.metadata}")
    print(f"Node ID: {sample_node.node_id}")
    print(f"Has prev node: {sample_node.prev_node is not None}")
    print(f"Has next node: {sample_node.next_node is not None}")

INFO:__main__:Loaded 4 documents from db
INFO:__main__:Chunking 4 documents...
INFO:__main__:Created 25 semantic chunks



Created 25 chunks from 4 documents

Sample chunk:
Text preview: file: A Brief History of Liberia and Africa's Iron Lady _ Ellen Johnson Sirleaf.mp3
language: en

 With its rocky history of violent civil wars, mili...
Metadata: {'documentary_name': "A Brief History of Liberia and Africa's Iron Lady _ Ellen Johnson Sirleaf.txt", 'source_id': 2, 'file_path': "/Users/antonio/Projekte/sofakles/Transkribtion/files/African politics/A Brief History of Liberia and Africa's Iron Lady _ Ellen Johnson Sirleaf.txt", 'relative_path': "African politics/A Brief History of Liberia and Africa's Iron Lady _ Ellen Johnson Sirleaf.txt", 'file_size': 15122, 'created_at': '2026-03-04 00:01:41.913866'}
Node ID: 41507c57-1576-4848-9e6c-2766ae5665c6
Has prev node: False
Has next node: True


In [18]:
from llama_index.core import VectorStoreIndex

index = VectorStoreIndex(nodes)

In [ ]:
from llama_index.llms.anthropic import Anthropic

llm = Anthropic(model="claude-sonnet-4-0", api_key="sk-ant-api03-your-key")

query_engine = index.as_query_engine(llm=llm, temperature=0.0)
response = query_engine.query("Who was Ellen Johnson?")

In [20]:
print(response)

Ellen Johnson Sirleaf was Africa's first elected female head of state who led Liberia through a challenging post-war period. She became president against a backdrop of sharp ethnic tensions and civil wars, successfully keeping her country together during this difficult transition.

During her presidency, she oversaw significant land concessions to multinational corporations, with over one-third of Liberia's landmass being awarded to foreign companies and more than 40% of the country's forests acquired by international logging firms. Her administration faced multiple corruption scandals that damaged her reputation as "the mother of the nation."

Despite controversies surrounding her corruption record and debates about her status as a feminist icon, she is recognized as a giant in modern African history for her role in stabilizing Liberia and guiding it through its post-conflict recovery period.


-------